# Surya OCR Pre-trained Model Benchmark

Benchmark the **Surya OCR** pre-trained model on the `avishadilhara/sinhala-ocr-lk-acts-1010` test dataset.

Surya supports 90+ languages including Sinhala, with line-level text detection and recognition.

In [ ]:
%%capture
# Pin transformers to 4.57.3 for surya-ocr compatibility
# (surya-ocr is incompatible with transformers 5.x)
!pip install -q transformers==4.57.3
!pip install -q surya-ocr
!pip install -q datasets
!pip install -q huggingface-hub>=1.3.1
!pip install -q jiwer
!pip install -q Pillow>=11.3.0
!pip install -q nltk

In [ ]:
import os
import sys
import json
import torch
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import pickle
from PIL import Image

from jiwer import cer, wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk

import warnings
warnings.filterwarnings('ignore')

try:
    nltk.data.find('wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('omw-1.4', quiet=True)


def sanitize_text(text):
    """Remove surrogate characters that cause UnicodeEncodeError in Jupyter."""
    if not isinstance(text, str):
        return text
    import re
    text = re.sub(r'[\ud800-\udfff]', '\ufffd', text)
    return text


class SafeStream:
    """Wrapper around stdout/stderr that strips surrogate characters."""
    def __init__(self, stream):
        self._stream = stream

    def write(self, text):
        if isinstance(text, str):
            text = sanitize_text(text)
        return self._stream.write(text)

    def flush(self):
        return self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


if not isinstance(sys.stdout, SafeStream):
    sys.stdout = SafeStream(sys.stdout)
if not isinstance(sys.stderr, SafeStream):
    sys.stderr = SafeStream(sys.stderr)

print("Safe stream wrappers installed.")

In [ ]:
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

print("=" * 80)
print("LOADING SURYA OCR PRE-TRAINED MODEL")
print("=" * 80)

print("\nLoading foundation predictor...")
foundation_predictor = FoundationPredictor()

print("Loading recognition predictor...")
recognition_predictor = RecognitionPredictor(foundation_predictor)

print("Loading detection predictor...")
detection_predictor = DetectionPredictor()

device = "cuda:0" if torch.cuda.is_available() else "cpu"

print("\nModel loaded successfully!")
print(f"Device: {device}")
print(f"Model: Surya OCR (pre-trained)")

In [ ]:
from datasets import load_dataset
from huggingface_hub import login

login()

In [ ]:
print("\n" + "="*80)
print("LOADING TEST DATASET")
print("="*80)

print("\nLoading dataset from Hugging Face...")
dataset = load_dataset("avishadilhara/sinhala-ocr-lk-acts-1010")

print(f"Dataset loaded!")
print(f"Test samples: {len(dataset['test'])}")

In [ ]:
CHECKPOINT_INTERVAL = 10
CHECKPOINT_FILE = "./test_results_surya/checkpoint.pkl"

In [ ]:
# STEP 3: DEFINE METRICS FUNCTIONS

def calculate_anls(ground_truth, prediction, threshold=0.5):
    """Calculate Average Normalized Levenshtein Similarity"""
    try:
        from Levenshtein import distance as levenshtein_distance

        if len(ground_truth) == 0 and len(prediction) == 0:
            return 1.0
        if len(ground_truth) == 0 or len(prediction) == 0:
            return 0.0

        edit_distance = levenshtein_distance(ground_truth, prediction)
        max_len = max(len(ground_truth), len(prediction))
        normalized_distance = edit_distance / max_len

        if normalized_distance < threshold:
            anls = 1 - normalized_distance
        else:
            anls = 0.0

        return anls
    except ImportError:
        from jiwer import cer
        return 1 - cer(ground_truth, prediction)


def calculate_all_metrics(ground_truth, prediction):
    """Calculate all metrics: CER, WER, BLEU, ANLS, METEOR"""
    ground_truth = sanitize_text(ground_truth)
    prediction = sanitize_text(prediction)

    metrics = {}

    try:
        raw_cer = cer(ground_truth, prediction)
        metrics['CER'] = min(raw_cer, 1.0)
    except:
        metrics['CER'] = 1.0

    try:
        raw_wer = wer(ground_truth, prediction)
        metrics['WER'] = min(raw_wer, 1.0)
    except:
        metrics['WER'] = 1.0

    try:
        reference = [list(ground_truth)]
        hypothesis = list(prediction)
        smoothing = SmoothingFunction().method1
        raw_bleu = sentence_bleu(reference, hypothesis,
                                        smoothing_function=smoothing)
        metrics['BLEU'] = max(0.0, min(raw_bleu, 1.0))
    except:
        metrics['BLEU'] = 0.0

    try:
        raw_anls = calculate_anls(ground_truth, prediction)
        metrics['ANLS'] = max(0.0, min(raw_anls, 1.0))
    except:
        metrics['ANLS'] = 0.0

    try:
        reference_tokens = list(ground_truth)
        hypothesis_tokens = list(prediction)
        raw_meteor = meteor_score([reference_tokens], hypothesis_tokens)
        metrics['METEOR'] = max(0.0, min(raw_meteor, 1.0))
    except:
        metrics['METEOR'] = 0.0

    metrics['Char_Accuracy'] = max((1 - metrics['CER']) * 100, 0.0)

    return metrics

In [ ]:
# CHECKPOINT MANAGEMENT

def save_checkpoint(all_results, year_results, last_processed_idx, output_dir):
    checkpoint_data = {
        'all_results': all_results,
        'year_results': dict(year_results),
        'last_processed_idx': last_processed_idx
    }

    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint_data, f)


def load_checkpoint(output_dir):
    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            checkpoint_data = pickle.load(f)
        return checkpoint_data
    return None

In [ ]:
# STEP 4: RUN TESTING ON ENTIRE TEST SET WITH CHECKPOINTING

def run_single_inference(image):
    predictions = recognition_predictor([image], det_predictor=detection_predictor)

    if predictions and len(predictions) > 0:
        page_result = predictions[0]
        text_lines = [sanitize_text(line.text) for line in page_result.text_lines]
        full_text = "\n".join(text_lines)
    else:
        full_text = ""

    return sanitize_text(full_text.strip())


def test_entire_dataset(resume=True):
    output_dir = "./test_results_surya"
    inference_dir = os.path.join(output_dir, "inference_outputs")
    os.makedirs(inference_dir, exist_ok=True)
    os.makedirs(os.path.join(inference_dir, "images"), exist_ok=True)

    all_results = []
    year_results = defaultdict(list)
    start_idx = 0

    if resume:
        checkpoint = load_checkpoint(output_dir)
        if checkpoint:
            all_results = checkpoint['all_results']
            year_results = defaultdict(list, checkpoint['year_results'])
            start_idx = checkpoint['last_processed_idx'] + 1

            print("=" * 80)
            print("CHECKPOINT FOUND - RESUMING FROM PREVIOUS RUN")
            print("=" * 80)
            print(f"Already processed: {len(all_results)} samples")
            print(f"Resuming from sample: {start_idx}")
            print("=" * 80)

    test_dataset = dataset['test']
    total_samples = len(test_dataset)

    if start_idx == 0:
        print("=" * 80)
        print("TESTING SURYA OCR MODEL ON ENTIRE TEST DATASET")
        print("=" * 80)
        print(f"Total samples to test: {total_samples}")
        print(f"Output directory: {inference_dir}")
        print(f"Checkpoint interval: Every {CHECKPOINT_INTERVAL} samples")
        print("=" * 80)

    samples_to_process = range(start_idx, total_samples)

    with tqdm(total=total_samples, initial=start_idx, desc="Testing", unit="sample", ncols=100) as pbar:
        for idx in samples_to_process:
            try:
                sample = test_dataset[idx]
                ground_truth = sanitize_text(sample['text'])
                year = sample.get('year', 'Unknown')
                image = sample['image']

                if image.mode == 'RGBA':
                    rgb_image = Image.new('RGB', image.size, (255, 255, 255))
                    rgb_image.paste(image, mask=image.split()[3])
                    image = rgb_image
                elif image.mode != 'RGB':
                    image = image.convert('RGB')

                img_filename = f"{idx:04d}.jpg"
                temp_img_path = os.path.join(inference_dir, "images", img_filename)
                image.save(temp_img_path)

                prediction = run_single_inference(image)

                pred_file = os.path.join(inference_dir, f"result_{idx:04d}.txt")
                with open(pred_file, 'w', encoding='utf-8') as f:
                    f.write(prediction)

                metrics = calculate_all_metrics(ground_truth, prediction)

                result = {
                    'sample_idx': idx,
                    'year': year,
                    'ground_truth_length': len(ground_truth),
                    'prediction_length': len(prediction),
                    **metrics
                }

                all_results.append(result)
                year_results[year].append(result)

                pbar.set_postfix({
                    'CER': f"{metrics['CER']:.3f}",
                    'Acc': f"{metrics['Char_Accuracy']:.1f}"
                })
                pbar.update(1)

                if (idx + 1) % CHECKPOINT_INTERVAL == 0:
                    save_checkpoint(all_results, year_results, idx, output_dir)
                    pbar.write(f"Checkpoint saved at sample {idx + 1}")

            except KeyboardInterrupt:
                print("\nTesting interrupted by user!")
                print("Saving checkpoint...")
                save_checkpoint(all_results, year_results, idx - 1, output_dir)
                print(f"Progress saved. Resume by running the code again.")
                print(f"Processed {len(all_results)}/{total_samples} samples")
                raise

            except Exception as e:
                error_msg = sanitize_text(str(e))[:80]
                pbar.write(f"Error at sample {idx}: {error_msg}...")

                all_results.append({
                    'sample_idx': idx,
                    'year': year if 'year' in dir() else 'Unknown',
                    'CER': 1.0,
                    'WER': 1.0,
                    'BLEU': 0.0,
                    'ANLS': 0.0,
                    'METEOR': 0.0,
                    'Char_Accuracy': 0.0,
                    'ground_truth_length': 0,
                    'prediction_length': 0
                })
                pbar.update(1)
                continue

    save_checkpoint(all_results, year_results, total_samples - 1, output_dir)
    print("\nAll samples processed!")

    return all_results, year_results

In [ ]:
# STEP 5: DISPLAY RESULTS

def display_results(all_results, year_results):
    print("\n" + "="*80)
    print("TEST RESULTS - SURYA OCR PRE-TRAINED MODEL")
    print("="*80)

    df = pd.DataFrame(all_results)

    print("\n" + "="*80)
    print("OVERALL AVERAGE METRICS")
    print("="*80)

    metrics_to_show = ['CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']

    for metric in metrics_to_show:
        if metric in df.columns:
            avg_value = df[metric].mean()

            if metric in ['CER', 'WER']:
                direction = "(lower is better)"
            else:
                direction = "(higher is better)"

            print(f"{metric:15}: {avg_value:7.4f}  {direction}")

    print(f"\nTotal Samples: {len(df)}")

    print("\n" + "="*80)
    print("YEAR-WISE AVERAGE METRICS")
    print("="*80)

    year_groups = df.groupby('year')
    year_summary = []

    for year, group in sorted(year_groups):
        year_data = {
            'Year': year,
            'Samples': len(group),
            'CER': f"{group['CER'].mean():.4f}",
            'WER': f"{group['WER'].mean():.4f}",
            'BLEU': f"{group['BLEU'].mean():.4f}",
            'ANLS': f"{group['ANLS'].mean():.4f}",
            'METEOR': f"{group['METEOR'].mean():.4f}",
            'Accuracy%': f"{group['Char_Accuracy'].mean():.2f}%"
        }
        year_summary.append(year_data)

    year_df = pd.DataFrame(year_summary)
    print("\n" + year_df.to_string(index=False))

    print("\n" + "="*80)
    print("INDIVIDUAL SAMPLE RESULTS (First 10 samples)")
    print("="*80)

    display_cols = ['sample_idx', 'year', 'CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']
    print("\n" + df[display_cols].head(10).to_string(index=False))

    print("\n" + "="*80)
    print("BEST PERFORMING SAMPLES (Top 5 by Character Accuracy)")
    print("="*80)

    best_samples = df.nlargest(5, 'Char_Accuracy')[display_cols]
    print("\n" + best_samples.to_string(index=False))

    print("\n" + "="*80)
    print("WORST PERFORMING SAMPLES (Bottom 5 by Character Accuracy)")
    print("="*80)

    worst_samples = df.nsmallest(5, 'Char_Accuracy')[display_cols]
    print("\n" + worst_samples.to_string(index=False))

    output_csv = "./test_results_surya/test_metrics.csv"
    df.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"\nFull results saved to: {output_csv}")

    year_csv = "./test_results_surya/year_wise_metrics.csv"
    year_df.to_csv(year_csv, index=False)
    print(f"Year-wise results saved to: {year_csv}")

    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)

    print(f"\nSamples with >90% accuracy: {len(df[df['Char_Accuracy'] > 90])}")
    print(f"Samples with >80% accuracy: {len(df[df['Char_Accuracy'] > 80])}")
    print(f"Samples with <50% accuracy: {len(df[df['Char_Accuracy'] < 50])}")

    print(f"\nMedian Character Accuracy: {df['Char_Accuracy'].median():.2f}%")
    print(f"Std Dev Character Accuracy: {df['Char_Accuracy'].std():.2f}%")

    return df, year_df

In [ ]:
# STEP 6: RUN TESTING

if __name__ == "__main__":
    print("\n" + "="*80)
    print("STARTING COMPREHENSIVE MODEL TESTING - SURYA OCR")
    print("="*80)

    try:
        all_results, year_results = test_entire_dataset(resume=True)
        results_df, year_df = display_results(all_results, year_results)

        print("\n" + "="*80)
        print("TESTING COMPLETED!")
        print("="*80)
        print(f"\nCheck ./test_results_surya/ directory for:")
        print(f"- test_metrics.csv (all samples)")
        print(f"- year_wise_metrics.csv (year summary)")
        print(f"- inference_outputs/ (model predictions as .txt files)")
        print(f"- checkpoint.pkl (resume checkpoint)")

    except KeyboardInterrupt:
        print("\nTesting paused. Run again to resume from checkpoint.")

In [ ]:
import shutil
import os
from IPython.display import FileLink

output_zip = 'test_results_surya.zip'

if os.path.exists('./test_results_surya'):
    if os.path.exists(output_zip):
        os.remove(output_zip)

    shutil.make_archive('test_results_surya', 'zip', './test_results_surya')

    print(f"Created {output_zip}")
    print(f"Size: {os.path.getsize(output_zip) / (1024*1024):.2f} MB")

    display(FileLink(output_zip))
else:
    print("test_results_surya folder not found")